[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leehart3000/zarrextract/blob/main/notebooks/benchmarks.ipynb)

# Benchmarks: zarrextract and xarray

This notebook times zarrextract against xarray on the same small selections from public Zarr stores, and checks that both return the same values. The notebook has no saved outputs: run it in Colab with Runtime, Run all. It reads from AWS Open Data and the Image Data Resource over the network, so the times depend on your connection, the Colab machine and the time of day. They change from one run to the next.

The question is a narrow one: you want a small piece of a large store, as values or as a table, and you want to know what it costs to get it. xarray does much more than that (labelled arrays, computation, lazy graphs), and this notebook does not measure any of it.

**Data and licences.**

- MUR sea surface temperature, from NASA JPL PO.DAAC, via the [Registry of Open Data on AWS](https://registry.opendata.aws/mur). No restrictions on use.
- CMIP6 `tas` (near-surface air temperature), historical experiment, member r1i1p1f1, from two models: MRI-ESM2-0 and NCAR CESM2. [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The [CMIP6 Terms of Use](https://pcmdi.llnl.gov/CMIP6/TermsOfUse) say that the CMIP6 licence page supersedes the `license` attribute in the files.
  - Yukimoto, S. et al. (2019). MRI MRI-ESM2.0 model output prepared for CMIP6 CMIP historical. Version 20190222. Earth System Grid Federation. https://doi.org/10.22033/ESGF/CMIP6.6842 (check the full author list at the DOI).
  - Danabasoglu, G. (2019). NCAR CESM2 model output prepared for CMIP6 CMIP historical. Version 20190308. Earth System Grid Federation. https://doi.org/10.22033/ESGF/CMIP6.7627 (check the full author list at the DOI).
  - As the Terms of Use ask, we acknowledge the World Climate Research Programme, which, through its Working Group on Coupled Modelling, coordinated and promoted CMIP6, the climate modelling groups (MRI and NCAR) who produced and made available their model output, and the Earth System Grid Federation (ESGF) for archiving the data and providing access.
- Light-sheet movie of a zebrafish embryo, Image Data Resource idr0051, Steventon et al., [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data DOI: https://doi.org/10.17867/10000117. Publication: Attardi A et al. (2018), Development, https://doi.org/10.1242/dev.166728. Also cite the Image Data Resource: Williams E. et al. (2017), Nature Methods 14, 775-781, https://doi.org/10.1038/nmeth.4326

## Install

Colab already has `fsspec`. `s3fs` needs the same version of it, so pin `s3fs` to that version and Colab's `fsspec` is not upgraded. `-U` brings zarrextract and xarray up to date, and xarray needs `cftime` to decode the `noleap` calendar of case 5. Colab normally has Dask, which the Dask contender needs; the versions printed below say what this run used. If Colab asks you to restart the session after the upgrade, restart it and run the notebook again from the next cell.

In [1]:
import fsspec  # noqa: F401 (used in the %pip line)

%pip install -q zarrextract "s3fs=={fsspec.__version__}" -U xarray cftime

In [2]:
import platform
import time
from collections.abc import Callable
from dataclasses import dataclass, field
from datetime import UTC, datetime
from importlib.metadata import PackageNotFoundError, version
from typing import Any

import numpy as np
import pyarrow as pa
import xarray as xr
import zarr

import zarrextract as zx

accessed = datetime.now(UTC).strftime("%Y-%m-%d")

print("Python", platform.python_version())
for package in (
    "zarrextract",
    "zarr",
    "xarray",
    "dask",
    "fsspec",
    "s3fs",
    "numpy",
    "pyarrow",
):
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "(not installed)")
print("Date", accessed)

print(
    "\nMulti-Scale Ultra High Resolution (MUR) Sea Surface Temperature (SST) "
    f"was accessed on {accessed} from https://registry.opendata.aws/mur"
)

Python 3.13.18
zarrextract 0.3.0
zarr 3.4.2
xarray 2026.9.0
dask 2026.8.0
fsspec 2025.12.0
s3fs 2025.12.0
numpy 2.4.6
pyarrow 23.0.1
Date 2026-11-27

Multi-Scale Ultra High Resolution (MUR) Sea Surface Temperature (SST) was accessed on 2026-11-27 from https://registry.opendata.aws/mur


## How each run is timed

A timed run does everything a user does to get the values: it opens the store, reads the coordinates it needs, applies the selection and reads the chunks. For xarray that is `xr.open_zarr(...)`, then the selection, then `.load()`. For zarrextract it is one call to `zx.extract(...)`.

Opening is also timed on its own, to show how much of a run goes on opening the store and reading its metadata. zarrextract has no separate open step, so its open time is the time of `zarr.open_group(...)` called as `extract()` calls it, which is the first thing `extract()` does. The open time is a separate run, not a part of the others, so the times in a row do not add up.

## Contenders and results

There are three contenders:

- `zarrextract`: `zx.extract(store, variable, storage_options=..., **selections)`.
- `xarray with Dask`: `xr.open_zarr(store, storage_options=...)`, then `.sel(...)`, then `.load()`. With the default `chunks` the variables are Dask arrays, so `.load()` runs a Dask graph.
- `xarray without Dask`: the same with `chunks=None`, which gives lazy arrays from xarray's own indexing and no Dask.

Each contender gives two results, each timed in a run of its own:

- **values in memory.** zarrextract: the Arrow table, the only thing it returns. xarray: the DataArray after `.load()`.
- **as a table.** zarrextract: the same Arrow table, so its two times are the same measurement. xarray: `.to_dataframe()` on the selected DataArray, a pandas DataFrame with one row per value. The time covers everything from opening the store to the finished table.

## Repetitions and correctness

Before anything is timed, a warm-up reads a little from each store (the S3 buckets and the HTTPS server), so that the first timed run is not the one that sets up the connection. The warm-up is not timed and uses `zarr` directly, so it does not warm either library's own code.

Every case then runs 5 times for each contender. The order of the contenders rotates in each repetition, and the contender that goes first also changes from one case to the next, so no contender always runs just after another has warmed something up. The table reports the first run, which is the cold one, and the median of the 5.

The cold run still includes what each library does only the first time it is called, such as its lazy imports. In case 1 zarrextract goes first, so read the cold column of that case with care. Nothing is cached on disk between runs, but fsspec reuses its client, so later runs reuse an open connection.

**Correctness is checked in every run.** Each result is compared with the values of the zarrextract table of the same repetition: the same number of values, in the same C order (the last dimension changes fastest). NaN in xarray and null in zarrextract count as equal. The verdict is one of:

- `identical`;
- `equal to float32 precision`: every value within a relative 1e-6. xarray can decode packed data to float32 where zarrextract gives float64;
- `different cell (zx lon X, xarray lon Y)`: in cases 4 and 5 the two libraries pick different grid cells (see below), so their values are not expected to agree. The values of each library are then compared with a direct `zarr` read of the cell it picked, and the verdict gives both results;
- `MISMATCH`, with the number of values that differ or the number of values returned.

A mismatch does not stop anything and is not hidden. The row stays in the table with its times and its verdict.

**A contender that cannot do a case** does not crash the notebook. The cell of the stage that failed shows `error: ` and the type of the exception, and that stage is not tried again for the case.

## Making the selections match

Both sides select by coordinate value. Where the two differ, this is how the notebook handles it.

- **Nearest values.** A scalar in zarrextract picks the nearest value, and the dimension stays in the table with length 1. For xarray we use `.sel(lat=[54.0], method="nearest")` with a one-element list, which also keeps the dimension. A bare scalar would drop it, and case 1 would then be a 0-dimensional array, which I expect `.to_dataframe()` to refuse. xarray's nearest selection never fails, however far the value is from the grid, while zarrextract raises an error when the value is more than one grid step outside the coordinates.
- **Longitude wraps in zarrextract only.** On a 0 to 360 grid, zarrextract finds the nearest column on the circle, so -0.1 can give the column at 0.0. xarray looks for the nearest label on the line, so 359.9 gives the largest longitude of the grid, which is more than a grid step from 0.0. In cases 4 and 5 the two libraries can therefore pick different cells, and the notebook reports `different cell` instead of a mismatch. It then reads each library's cell directly with `zarr` and checks that library's values against it. For case 4 the direct read takes every time step; for case 5 it takes the last 12 (the historical run ends in December 2014).
- **Ranges are slices.** xarray's `slice(start, stop)` is by label and includes both ends, as in zarrextract. A time slice given as a string covers the whole period, as it does in zarrextract: `slice("2019-06", "2019-06")` is all of June 2019, and a stop of `"2019-06-30"` includes every time on that day. xarray does not let `method` be combined with a slice, so each selection is done in two steps, slices first and nearest values second.
- **Slice order and bounds.** xarray's slice follows the order of the index: on an ascending axis a start above the stop gives an empty selection. It does not wrap a longitude range around the seam of a 0 to 360 grid. Case 6 therefore uses the usual idiom: relabel the longitude to -180 to 180 with `assign_coords`, sort by it with `sortby("lon")`, then select. The rows then come in a different order from zarrextract, which lists 349 to 359 and then 0 to 2, so both results are sorted by time, latitude and longitude before the values are compared.
- **No coordinates.** The IDR image has no coordinate arrays, so its dimensions have no index in xarray and `.sel` does not apply. xarray selects by position with `.isel(c=0, z=100, y=150, x=100)`.

## The cases

Each case holds the selections for zarrextract and the matching ones for xarray.

1. MUR, one point and one time (nearest).
2. MUR, the same point, the whole of June 2019.
3. MUR, a box (50 to 52 N, -2 to 2 E) at one time.
4. CMIP6 MRI-ESM2-0, `tas` at London over 1850 to 2014 (1980 months). xarray: longitude 359.9.
5. CMIP6 CESM2, `tas` at London in 2014. The calendar is `noleap`, so xarray decodes the time with cftime.
6. MRI-ESM2-0, `tas` in a box across the prime meridian (49 to 61 N, -11 to 2 E), July 2014.
7. IDR OME-Zarr image, array `0/0`: channel 0, plane 100, pixel y=150, x=100, at all time points. xarray tries to open the group `0`, which may fail because the server cannot list folders; the error type is recorded. If it opens, it selects by position with `.isel`.
8. MUR, the box of case 3 for the whole of June 2019, a larger read.

In [3]:
S3 = {"anon": True}  # public buckets: no credentials

MUR = "s3://mur-sst/zarr-v1"
MRI = (
    "s3://cmip6-pds/CMIP6/CMIP/MRI/MRI-ESM2-0/historical/"
    "r1i1p1f1/Amon/tas/gn/v20190222/"
)
CESM2 = (
    "s3://cmip6-pds/CMIP6/CMIP/NCAR/CESM2/historical/r1i1p1f1/Amon/tas/gn/v20190308/"
)
IDR = (
    "https://livingobjects.ebi.ac.uk/idr/zarr/v0.5/idr0051/"
    "180712_H2B_22ss_Courtney1_20180712-163837_p00_c00_preview.zarr"
)


def longitude_180(longitude):
    """Relabel longitudes from 0 to 360 to -180 to 180."""
    return (longitude + 180) % 360 - 180


@dataclass(frozen=True)
class Case:
    number: int
    title: str
    store: str
    variable: str  # the array, as zarrextract names it
    zx: dict[str, Any]  # the zarrextract selections
    options: dict[str, Any] | None = None  # storage_options
    xr_variable: str | None = None  # the array, as xarray names it
    xr_open: dict[str, Any] = field(default_factory=dict)  # for open_zarr
    xr_ranges: dict[str, Any] = field(default_factory=dict)  # slices
    xr_points: dict[str, Any] = field(default_factory=dict)  # nearest
    xr_positions: dict[str, int] = field(default_factory=dict)  # isel, no index
    relabel_lon: bool = False  # xarray: longitude to -180..180, then sorted
    sort_by: tuple[str, ...] = ()  # sort both results by these dimensions
    # One cell, as (time, lat, lon): the times a direct read of the cell takes.
    # Set it to compare each library's own cell when they may pick different cells.
    direct_time: slice | None = None

    @property
    def xr_name(self) -> str:
        return self.xr_variable or self.variable


CASES = [
    Case(
        1,
        "MUR, one point",
        MUR,
        "analysed_sst",
        zx={"lat": 54.0, "lon": 3.0, "time": "2019-06-01"},
        options=S3,
        xr_points={
            "lat": [54.0],
            "lon": [3.0],
            "time": [np.datetime64("2019-06-01")],
        },
    ),
    Case(
        2,
        "MUR, one point, June 2019",
        MUR,
        "analysed_sst",
        zx={"lat": 54.0, "lon": 3.0, "time": ("2019-06", "2019-06")},
        options=S3,
        xr_ranges={"time": slice("2019-06", "2019-06")},
        xr_points={"lat": [54.0], "lon": [3.0]},
    ),
    Case(
        3,
        "MUR, box, one day",
        MUR,
        "analysed_sst",
        zx={"lat": (50, 52), "lon": (-2, 2), "time": "2019-06-01"},
        options=S3,
        xr_ranges={"lat": slice(50, 52), "lon": slice(-2, 2)},
        xr_points={"time": [np.datetime64("2019-06-01")]},
    ),
    Case(
        4,
        "MRI-ESM2-0, London, 1850 to 2014",
        MRI,
        "tas",
        zx={"lat": 51.5, "lon": -0.1, "time": ("1850-01-01", "2014-12-31")},
        options=S3,
        xr_ranges={"time": slice("1850-01-01", "2014-12-31")},
        xr_points={"lat": [51.5], "lon": [359.9]},
        direct_time=slice(None),  # the whole run
    ),
    Case(
        5,
        "CESM2, London, 2014 (noleap)",
        CESM2,
        "tas",
        zx={"lat": 51.5, "lon": -0.1, "time": ("2014-01-01", "2014-12-31")},
        options=S3,
        xr_ranges={"time": slice("2014-01-01", "2014-12-31")},
        xr_points={"lat": [51.5], "lon": [359.9]},
        direct_time=slice(-12, None),  # the run ends in December 2014
    ),
    Case(
        6,
        "MRI-ESM2-0, box across 0 E, July 2014",
        MRI,
        "tas",
        zx={"lat": (49, 61), "lon": (-11, 2), "time": ("2014-07-01", "2014-07-31")},
        options=S3,
        xr_ranges={
            "lat": slice(49, 61),
            "lon": slice(-11, 2),
            "time": slice("2014-07-01", "2014-07-31"),
        },
        relabel_lon=True,
        sort_by=("time", "lat", "lon"),
    ),
    Case(
        7,
        "IDR OME-Zarr, one pixel, all times",
        IDR,
        "0/0",
        zx={"c": 0, "z": 100, "y": 150, "x": 100},
        xr_variable="0",
        xr_open={"group": "0"},
        xr_positions={"c": 0, "z": 100, "y": 150, "x": 100},
    ),
    Case(
        8,
        "MUR, box, June 2019",
        MUR,
        "analysed_sst",
        zx={"lat": (50, 52), "lon": (-2, 2), "time": ("2019-06-01", "2019-06-30")},
        options=S3,
        xr_ranges={
            "lat": slice(50, 52),
            "lon": slice(-2, 2),
            "time": slice("2019-06-01", "2019-06-30"),
        },
    ),
]

## The contenders

Each contender has three functions: one that only opens the store, one that gets the values, and (for xarray) one that gets a table.

In [4]:
@dataclass(frozen=True)
class Contender:
    name: str
    open: Callable[[Case], Any]
    values: Callable[[Case], Any]
    table: Callable[[Case], Any] | None = None  # None: the values are the table


def zx_open(case: Case) -> Any:
    # What extract() does first. zarrextract has no public function to open.
    return zarr.open_group(
        case.store,
        mode="r",
        use_consolidated=None,
        storage_options=case.options,
    )


def zx_extract(case: Case) -> pa.Table:
    return zx.extract(
        case.store, case.variable, storage_options=case.options, **case.zx
    )


def select_xarray(dataset: xr.Dataset, case: Case) -> xr.DataArray:
    """Select with xarray as the case describes. Nothing is read yet."""
    if case.relabel_lon:
        dataset = dataset.assign_coords(lon=longitude_180(dataset["lon"]))
        dataset = dataset.sortby("lon")
    array = dataset[case.xr_name]
    if case.xr_ranges:
        array = array.sel(case.xr_ranges)
    if case.xr_points:
        array = array.sel(case.xr_points, method="nearest")
    if case.xr_positions:
        array = array.isel(case.xr_positions)
    return array


def xarray_contender(name: str, **open_options: Any) -> Contender:
    def open_store(case: Case) -> xr.Dataset:
        return xr.open_zarr(
            case.store, storage_options=case.options, **case.xr_open, **open_options
        )

    def values(case: Case) -> xr.DataArray:
        return select_xarray(open_store(case), case).load()

    def table(case: Case) -> Any:
        return select_xarray(open_store(case), case).to_dataframe()

    return Contender(name, open_store, values, table)


CONTENDERS = [
    Contender("zarrextract", zx_open, zx_extract),
    xarray_contender("xarray with Dask"),
    xarray_contender("xarray without Dask", chunks=None),
]

## Timing and checking

`values_of()` turns any result into its values in row order, sorted by the case's `sort_by` dimensions if it has some. `compare()` gives the verdict for two arrays. For the cases with a `direct_time`, `cell_of()` finds the grid cell of a result, and `judge()` says `different cell` when the two libraries picked different cells, after checking each library's values against `direct_values()`, a direct read of its own cell. `attempt()` times one stage and turns an exception into an error type.

In [5]:
REPETITIONS = 5


def timed(function: Callable[[Case], Any], case: Case) -> tuple[float, Any]:
    start = time.perf_counter()
    result = function(case)
    return time.perf_counter() - start, result


def values_of(result: Any, case: Case) -> np.ndarray:
    """Return the values of a result in row order, sorted if the case says so."""
    keys: dict[str, np.ndarray] = {}
    if isinstance(result, pa.Table):
        values = result[case.variable].to_numpy()
        keys = {name: result[name].to_numpy() for name in case.sort_by}
        if case.relabel_lon:
            keys["lon"] = longitude_180(keys["lon"])
    elif isinstance(result, xr.DataArray):
        values = result.values.ravel()
        if case.sort_by:
            grids = np.meshgrid(
                *(result[dim].values for dim in result.dims), indexing="ij"
            )
            keys = {
                dim: grid.ravel() for dim, grid in zip(result.dims, grids, strict=True)
            }
    else:  # the DataFrame from .to_dataframe()
        values = result[case.xr_name].to_numpy()
        keys = {
            name: result.index.get_level_values(name).to_numpy()
            for name in case.sort_by
        }
    if case.sort_by:
        order = np.lexsort([keys[name] for name in reversed(case.sort_by)])
        values = values[order]
    return values


def cell_of(result: Any) -> tuple[float, float]:
    """Return the (lat, lon) of the first row of a result."""
    if isinstance(result, pa.Table):
        return float(result["lat"][0].as_py()), float(result["lon"][0].as_py())
    if isinstance(result, xr.DataArray):
        return float(result["lat"].values.ravel()[0]), float(
            result["lon"].values.ravel()[0]
        )
    index = result.index  # the DataFrame from .to_dataframe()
    return float(index.get_level_values("lat")[0]), float(
        index.get_level_values("lon")[0]
    )


DIRECT: dict[tuple[int, float, float], np.ndarray] = {}


def direct_values(case: Case, cell: tuple[float, float]) -> np.ndarray:
    """Read one grid cell straight from the store with zarr (read once)."""
    key = (case.number, *cell)
    if key not in DIRECT:
        group = zarr.open_group(case.store, mode="r", storage_options=case.options)
        lat_index = int(np.argmin(np.abs(group["lat"][:] - cell[0])))
        lon_index = int(np.argmin(np.abs(group["lon"][:] - cell[1])))
        array = group[case.variable]
        DIRECT[key] = np.asarray(array[case.direct_time, lat_index, lon_index])
    return DIRECT[key]


def compare(got: np.ndarray, expected: np.ndarray) -> str:
    """Say whether two arrays of values agree, treating NaN as equal to NaN."""
    if got.shape != expected.shape:
        return f"MISMATCH: {got.size} values, expected {expected.size}"
    if np.array_equal(got, expected, equal_nan=True):
        return "identical"
    close = np.isclose(
        got.astype("float64"),
        expected.astype("float64"),
        rtol=1e-6,
        atol=0,
        equal_nan=True,
    )
    if close.all():
        return "equal to float32 precision"
    return f"MISMATCH: {(~close).sum()} of {got.size} values differ"


@dataclass
class Converted:
    values: np.ndarray
    cell: tuple[float, float] | None = None  # for the cases with a direct_time


@dataclass
class Record:
    open: list[float] = field(default_factory=list)  # seconds, one per run
    values: list[float] = field(default_factory=list)
    table: list[float] = field(default_factory=list)
    errors: dict[str, str] = field(default_factory=dict)  # stage: error type
    verdicts: dict[str, list[str]] = field(default_factory=dict)  # per stage
    rows: int | None = None


def attempt(
    record: Record,
    stage: str,
    function: Callable[[Case], Any],
    case: Case,
    *,
    convert: bool = False,
) -> Converted | None:
    """Time one stage. Keep an error's type instead of raising it.

    A stage that has failed is not tried again. If ``convert`` is true and the
    stage worked, return the values of its result (and its cell, for the cases
    that compare cells), otherwise return ``None``. The conversion is not timed.
    """
    if stage in record.errors:
        return None
    try:
        seconds, result = timed(function, case)
        converted = None
        if convert:
            cell = None if case.direct_time is None else cell_of(result)
            converted = Converted(values_of(result, case), cell)
    except Exception as error:
        record.errors[stage] = type(error).__name__
        return None
    getattr(record, stage).append(seconds)
    return converted


def judge(case: Case, got: Converted, reference: Converted) -> str:
    """Give the verdict on a result, against the zarrextract result."""
    if got.cell is None or reference.cell is None:
        return compare(got.values, reference.values)
    if np.allclose(got.cell, reference.cell):
        return compare(got.values, reference.values)
    zx_check = compare(reference.values, direct_values(case, reference.cell))
    xarray_check = compare(got.values, direct_values(case, got.cell))
    return (
        f"different cell (zx lon {reference.cell[1]:g}, xarray lon {got.cell[1]:g}); "
        f"zx values vs direct read: {zx_check}; "
        f"xarray values vs direct read: {xarray_check}"
    )


def check(
    case: Case,
    reference_name: str,
    found: dict[tuple[str, str], Converted],
    records: dict[tuple[int, str], Record],
) -> None:
    """Compare each result with the zarrextract values and keep the verdicts."""
    reference = found.get((reference_name, "values"))
    for (name, stage), got in found.items():
        verdicts = records[case.number, name].verdicts.setdefault(stage, [])
        if name == reference_name:
            verdicts.append("reference")
        elif reference is None:
            verdicts.append("no reference (zarrextract failed)")
        else:
            try:
                verdicts.append(judge(case, got, reference))
            except Exception as error:
                verdicts.append(f"not compared ({type(error).__name__})")

## Run

First an untimed warm-up reads a little from each store: it opens the group and reads one value of the first coordinate it finds (the IDR store has none, so only its metadata is read). Then every case runs 5 times for each of 3 contenders, with 3 stages each, which can take a while. One dot is printed per repetition.

In [6]:
def warm_up(cases: list[Case]) -> None:
    """Read a little from each store once, untimed, to set up the connections."""
    seen = set()
    for case in cases:
        if case.store in seen:
            continue
        seen.add(case.store)
        try:
            group = zarr.open_group(case.store, mode="r", storage_options=case.options)
            array = group[case.variable]
            dims = getattr(array.metadata, "dimension_names", None)
            for dim in dims or array.attrs.get("_ARRAY_DIMENSIONS", []):
                if dim in group:
                    group[dim][:1]
                    break
            print("Warm-up done:", case.store)
        except Exception as error:
            print("Warm-up failed:", case.store, type(error).__name__)


def run(
    cases: list[Case],
    contenders: list[Contender],
    repetitions: int = REPETITIONS,
) -> dict[tuple[int, str], Record]:
    records = {
        (case.number, contender.name): Record()
        for case in cases
        for contender in contenders
    }
    for index, case in enumerate(cases):
        print(f"Case {case.number}: {case.title} ", end="")
        for repetition in range(repetitions):
            # Rotate the order in each repetition and from one case to the next.
            shift = (index + repetition) % len(contenders)
            found: dict[tuple[str, str], Converted] = {}
            for contender in contenders[shift:] + contenders[:shift]:
                record = records[case.number, contender.name]
                attempt(record, "open", contender.open, case)
                values = attempt(record, "values", contender.values, case, convert=True)
                if values is not None:
                    found[contender.name, "values"] = values
                    record.rows = values.values.size
                if contender.table is None:
                    if values is not None:  # the same run is the table
                        record.table.append(record.values[-1])
                    continue
                values = attempt(record, "table", contender.table, case, convert=True)
                if values is not None:
                    found[contender.name, "table"] = values
            check(case, contenders[0].name, found, records)
            print(".", end="", flush=True)
        print()
    return records


warm_up(CASES)
records = run(CASES, CONTENDERS)

Warm-up done: s3://mur-sst/zarr-v1
Warm-up done: s3://cmip6-pds/CMIP6/CMIP/MRI/MRI-ESM2-0/historical/r1i1p1f1/Amon/tas/gn/v20190222/
Warm-up done: s3://cmip6-pds/CMIP6/CMIP/NCAR/CESM2/historical/r1i1p1f1/Amon/tas/gn/v20190308/
Warm-up done: https://livingobjects.ebi.ac.uk/idr/zarr/v0.5/idr0051/180712_H2B_22ss_Courtney1_20180712-163837_p00_c00_preview.zarr
Case 1: MUR, one point .....
Case 2: MUR, one point, June 2019 .....
Case 3: MUR, box, one day .....
Case 4: MRI-ESM2-0, London, 1850 to 2014 .....
Case 5: CESM2, London, 2014 (noleap) .....
Case 6: MRI-ESM2-0, box across 0 E, July 2014 .....
Case 7: IDR OME-Zarr, one pixel, all times 

/tmp/ipykernel_2689/2601173463.py:42: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  return xr.open_zarr(


.....
Case 8: MUR, box, June 2019 .....


## Results

One row per case and contender. All times are in seconds.

- **open**: the median of the 5 runs that only open the store.
- **cold**: the first run of the values (open and read), after the warm-up but before anything else is warm.
- **median, values** and **median, as a table**: the medians of the 5 runs of each. For zarrextract they are the same runs.
- **rows**: the number of values returned.
- **correctness**: the verdict against the zarrextract values (see above), including `different cell` for cases 4 and 5. `error: ` followed by a type means the stage failed with that exception.

In [7]:
def shown(record: Record, stage: str, pick: Callable[[list[float]], float]) -> str:
    if stage in record.errors:
        return f"error: {record.errors[stage]}"
    times = getattr(record, stage)
    return f"{pick(times):.3g}" if times else "-"


def correctness(record: Record) -> str:
    texts = {
        stage: "; ".join(dict.fromkeys(found))
        for stage, found in record.verdicts.items()
    }
    if not texts:
        return "-"
    if len(set(texts.values())) == 1:
        return next(iter(texts.values()))
    return "; ".join(f"{stage}: {text}" for stage, text in texts.items())


def results_table(
    cases: list[Case],
    contenders: list[Contender],
    records: dict[tuple[int, str], Record],
) -> list[str]:
    """Return the lines of the results as a Markdown table."""
    columns = [
        "case",
        "contender",
        "open (s)",
        "cold (s)",
        "median, values (s)",
        "median, as a table (s)",
        "rows",
        "correctness",
    ]
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for case in cases:
        for contender in contenders:
            record = records[case.number, contender.name]
            cells = [
                f"{case.number}. {case.title}",
                contender.name,
                shown(record, "open", np.median),
                shown(record, "values", lambda times: times[0]),
                shown(record, "values", np.median),
                shown(record, "table", np.median),
                "-" if record.rows is None else f"{record.rows:,}",
                correctness(record),
            ]
            lines.append("| " + " | ".join(cells) + " |")
    return lines


for line in results_table(CASES, CONTENDERS, records):
    print(line)

| case | contender | open (s) | cold (s) | median, values (s) | median, as a table (s) | rows | correctness |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1. MUR, one point | zarrextract | 0.0541 | 1.6 | 1.04 | 1.04 | 1 | reference |
| 1. MUR, one point | xarray with Dask | 5.88 | 7.91 | 7.52 | 7.32 | 1 | identical |
| 1. MUR, one point | xarray without Dask | 5.92 | 6.3 | 6.23 | 6.13 | 1 | identical |
| 2. MUR, one point, June 2019 | zarrextract | 0.0476 | 1.56 | 1.88 | 1.88 | 30 | reference |
| 2. MUR, one point, June 2019 | xarray with Dask | 5.79 | 12.4 | 9.54 | 9.57 | 30 | identical |
| 2. MUR, one point, June 2019 | xarray without Dask | 5.82 | 6.68 | 6.68 | 6.93 | 30 | identical |
| 3. MUR, box, one day | zarrextract | 0.0466 | 1.55 | 1.11 | 1.11 | 80,601 | reference |
| 3. MUR, box, one day | xarray with Dask | 5.78 | 6.48 | 6.48 | 6.62 | 80,601 | identical |
| 3. MUR, box, one day | xarray without Dask | 5.76 | 6.79 | 6.44 | 6.31 | 80,601 | identical |
| 4. MRI-ESM2-0, 

## Repeated selections on one open store

The runs above open the store every time, so they measure one selection from a cold start. This section measures what a user does with a store they keep: open it once, then make several different selections from it.

There are two contenders:

- `zx.open, repeated`: `zx.open(store, storage_options=...)` once, then `src.extract(variable, **selections)` for each selection.
- `xarray without Dask, already open`: `xr.open_zarr(store, storage_options=..., chunks=None)` once, then `.sel(...)` and `.load()` for each selection.

In each repetition the store is opened once (a new handle, so nothing is kept from the last repetition). The case's own selection is timed first, then `SELECTIONS` = 5 different ones, which are the case's selection moved a little (see `shifted()`): a point in time moves by one day each, a time range by 30 days each, and cases 4 and 5 move the latitude by one degree each. Case 7 moves to the next plane and only zarrextract can do it. The two libraries get the same selection each time.

Every result is checked against a one-shot `zx.extract()` of the same selection, with the same verdicts as above. The reference is not timed. The table gives the median open time, the median time of the first selection, and the median time of the further selections. The section needs a zarrextract that has `zx.open()`, and says so if yours does not. It takes several minutes, because xarray opens each MUR store in each repetition.

In [ ]:
from dataclasses import replace

SELECTIONS = 5  # further selections on one open store, after the first


def shift_text(text: str, k: int, days: int) -> str:
    """Move an ISO date by k steps: months or years as written, else k * days."""
    moment = np.datetime64(text)
    unit = np.datetime_data(moment.dtype)[0]
    step = k if unit in ("Y", "M") else k * days
    return str(moment + np.timedelta64(step, unit))


def shifted(case: Case, k: int) -> Case:
    """Return the case with its k-th different selection.

    A case with a direct read (4 and 5) moves its latitude by k degrees, so that
    the time range, and with it the direct read, stays the same. A point in time
    moves by k days and a time range by 30 * k days (or k months, if it is given
    in months). The IDR image moves to plane 100 + k. Both libraries get the
    same selection.
    """
    if k == 0:
        return case
    if case.direct_time is not None:
        lat = case.zx["lat"] + k
        return replace(
            case,
            zx={**case.zx, "lat": lat},
            xr_points={**case.xr_points, "lat": [lat]},
        )
    if "z" in case.zx:
        z = case.zx["z"] + k
        return replace(
            case,
            zx={**case.zx, "z": z},
            xr_positions={**case.xr_positions, "z": z},
        )
    when = case.zx["time"]
    if isinstance(when, tuple):
        moved = tuple(shift_text(text, k, 30) for text in when)
        return replace(
            case,
            zx={**case.zx, "time": moved},
            xr_ranges={**case.xr_ranges, "time": slice(*moved)},
        )
    moved_text = shift_text(when, k, 1)
    return replace(
        case,
        zx={**case.zx, "time": moved_text},
        xr_points={**case.xr_points, "time": [np.datetime64(moved_text)]},
    )


@dataclass(frozen=True)
class Repeater:
    name: str
    open: Callable[[Case], Any]  # open the store once
    select: Callable[[Any, Case], Any]  # one selection on the open store


def zx_handle(case: Case) -> Any:
    return zx.open(case.store, storage_options=case.options)


def zx_select(handle: Any, case: Case) -> pa.Table:
    return handle.extract(case.variable, **case.zx)


def xr_handle(case: Case) -> xr.Dataset:
    return xr.open_zarr(
        case.store, storage_options=case.options, chunks=None, **case.xr_open
    )


def xr_select(dataset: xr.Dataset, case: Case) -> xr.DataArray:
    # The relabel and sort of case 6 are part of each timed selection.
    return select_xarray(dataset, case).load()


REPEATERS = [
    Repeater("zx.open, repeated", zx_handle, zx_select),
    Repeater("xarray without Dask, already open", xr_handle, xr_select),
]
REPEATED_CASES = CASES


@dataclass
class RepeatedRecord:
    open: list[float] = field(default_factory=list)  # seconds, one per repetition
    first: list[float] = field(default_factory=list)  # the case's own selection
    further: list[float] = field(default_factory=list)  # the shifted selections
    verdicts: list[str] = field(default_factory=list)
    error: str | None = None  # the type of the exception that stopped it


def as_converted(result: Any, case: Case) -> Converted:
    cell = None if case.direct_time is None else cell_of(result)
    return Converted(values_of(result, case), cell)


def run_repeated(
    cases: list[Case],
    repeaters: list[Repeater],
    repetitions: int = REPETITIONS,
) -> dict[tuple[int, str], RepeatedRecord]:
    """Open each store once per repetition, then time several selections on it.

    Each result is checked against a one-shot ``zx.extract()`` of the same
    selection. That reference is made once and is not timed.
    """
    records = {
        (case.number, repeater.name): RepeatedRecord()
        for case in cases
        for repeater in repeaters
    }
    references: dict[tuple[int, int], Converted] = {}
    for index, case in enumerate(cases):
        print(f"Case {case.number}: {case.title} ", end="")
        for repetition in range(repetitions):
            shift = (index + repetition) % len(repeaters)
            for repeater in repeaters[shift:] + repeaters[:shift]:
                record = records[case.number, repeater.name]
                if record.error is not None:
                    continue
                handle = None
                try:
                    seconds, handle = timed(repeater.open, case)
                    record.open.append(seconds)
                    for k in range(SELECTIONS + 1):
                        one = shifted(case, k)
                        if (case.number, k) not in references:
                            references[case.number, k] = as_converted(
                                zx_extract(one), one
                            )
                        start = time.perf_counter()
                        result = repeater.select(handle, one)
                        seconds = time.perf_counter() - start
                        (record.first if k == 0 else record.further).append(seconds)
                        record.verdicts.append(
                            judge(
                                one,
                                as_converted(result, one),
                                references[case.number, k],
                            )
                        )
                except Exception as error:
                    record.error = type(error).__name__
                finally:
                    close = getattr(handle, "close", None)
                    if close is not None:
                        close()
            print(".", end="", flush=True)
        print()
    return records


repeated_records: dict[tuple[int, str], RepeatedRecord] = {}
if hasattr(zx, "open"):
    repeated_records = run_repeated(REPEATED_CASES, REPEATERS)
else:
    print(
        f"zarrextract {zx.__version__} has no zx.open(). "
        "Install a newer release to run this section."
    )

In [ ]:
def median_or_dash(times: list[float]) -> str:
    return f"{np.median(times):.3g}" if times else "-"


def repeated_table(
    cases: list[Case],
    repeaters: list[Repeater],
    records: dict[tuple[int, str], RepeatedRecord],
) -> list[str]:
    """Return the lines of the repeated-selection results as a Markdown table."""
    columns = [
        "case",
        "contender",
        "open (s)",
        "first selection (s)",
        "per further selection (s)",
        "correctness",
    ]
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for case in cases:
        for repeater in repeaters:
            record = records[case.number, repeater.name]
            if record.error is not None:
                cells = [f"error: {record.error}"] * 3 + ["-"]
            else:
                cells = [
                    median_or_dash(record.open),
                    median_or_dash(record.first),
                    median_or_dash(record.further),
                    "; ".join(dict.fromkeys(record.verdicts)) or "-",
                ]
            row = [f"{case.number}. {case.title}", repeater.name, *cells]
            lines.append("| " + " | ".join(row) + " |")
    return lines


if repeated_records:
    for line in repeated_table(REPEATED_CASES, REPEATERS, repeated_records):
        print(line)

The first-selection column includes the reads that fill the caches (zarrextract) or the first lazy read (xarray). The further-selection column is the median over all `SELECTIONS` later selections in all repetitions. Compare it with the median, values column of the table above, which is the cost of one `zx.extract()` call that opens the store each time. This section has no saved output yet: its numbers will be added to the Performance page after a run in Colab.

## What the numbers mean

- **Long, finely chunked coordinates are where ZarrExtract gains most.** MUR SST stores its 6,443 days in chunks of 5, so xarray spends about 5.8 seconds opening the store, reading every coordinate to build its indexes. ZarrExtract opens the store in about 0.05 seconds and finds the dates it needs by bisection, reading a few chunks instead of all of them.
- **With short coordinates, the gap is small.** The CMIP6 stores open in about 0.35 seconds in xarray, and ZarrExtract is 1.2 to 1.8 times faster.
- **Reading the data itself is not faster.** For the larger read (case 8), xarray needs about 2.6 seconds after opening, the same as ZarrExtract in total. The difference is in finding the data, not in reading it.
- **Many small selections from one open dataset favour xarray.** Once a dataset is open, each further xarray selection in cases 1 to 3 took 0.3 to 0.9 seconds. That figure was derived from the totals above (the median time minus the time to open the store), not measured on its own. The timings above are for `zx.extract()`, which opens the store on every call. `zx.open()` keeps the store, the metadata and the coordinates it has read between calls, but these timings were taken before it existed. The section on repeated selections times both libraries on an open store, and the numbers will be added to the Performance page after it has been run.
- **Dask adds overhead to small reads.** xarray was slower with Dask than without it in every case here.
- **Longitude across the seam.** For London (longitude -0.1) on stores that run from 0 to 360, xarray's `method="nearest"` picked 358.9° or 358.8°, about 1° away, because it doesn't wrap around. ZarrExtract picked 0°, 0.1° away.
- **Stores that can't be listed.** xarray couldn't open the IDR image, because the web server refuses to list folders. ZarrExtract reads it with a known array path.